# Recovery Funnel EDA — Site & GL Characterisation

**Framing.** "Recovery" in this dataset means a unit *entered the recovery funnel* (recycling, donations, liquidations, return-to-vendor, etc.) instead of being sold through the normal channel. **A higher funnel-entry rate is a worse business outcome.** The goal of this notebook is to help stakeholders identify which sites and GL product groups are sending the most inventory into the funnel and to understand what characteristics and seasonal patterns drive high rates — so that interventions can be prioritised.

**Scope.** Row-level raw data → site × GL × week aggregation → share and COGS features (no temporal lag/rolling features) → EDA with seasonal decomposition.

In [ ]:
!pip install polars --quiet

In [ ]:
import polars as pl
import polars.selectors as cs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch

pl.Config.set_tbl_rows(30)
sns.set_theme(style='whitegrid', font_scale=0.95)

BAD_COLOR  = '#D7472E'   # red — high funnel-entry (bad)
GOOD_COLOR = '#2A9D5F'   # green — low funnel-entry (good)
NEU_COLOR  = '#5B8DB8'   # blue — neutral / predicted

OUTCOME_COLS = [
    'units_remove_return', 'units_donations', 'units_bintool_donations',
    'units_warehouse_deals_and_gr', 'units_liquidations',
    'units_return_to_vendor', 'units_bintool_theft',
    'units_remove_liquidate', 'units_bintool_remove_liquidate',
]
OUTCOME_LABELS = [
    'Remove/Return', 'Donations', 'Bintool Donations',
    'Warehouse Deals', 'Liquidations',
    'Return to Vendor', 'Bintool Theft',
    'Remove Liquidate', 'Bintool Remove Liq.',
]
OUTCOME_COLORS = sns.color_palette('tab10', n_colors=len(OUTCOME_COLS))


## 1. Load raw data

In [ ]:
df_raw = pl.read_parquet("s3://msds-26.2-data/clean/combined_recovery_data.parquet")
df_raw = df_raw.filter(
    pl.col('gl_product_group').is_not_null()
    & (pl.col('gl_product_group').cast(pl.Utf8) != '-1')
)
print('shape:', df_raw.shape)
print('columns:', df_raw.columns)
df_raw.head(3)


In [ ]:
print('recovery_type value counts:')
print(df_raw['recovery_type'].value_counts().sort('count', descending=True))
print('\nyear range:', df_raw['year'].min(), '–', df_raw['year'].max())
print('unique sites:', df_raw['hashed_fc'].n_unique())
print('unique GLs:', df_raw['gl_product_group'].n_unique())


## 2. Aggregate to site × GL × week

In [ ]:
def safe_sum_when(condition, col='units'):
    return pl.when(condition).then(pl.col(col)).otherwise(0).sum()

agg = (
    df_raw.group_by(['hashed_fc', 'year', 'month', 'week', 'gl_product_group'])
    .agg([
        # volume
        pl.col('units').sum().alias('units_total'),
        pl.col('cogs').sum().alias('cogs_total'),
        pl.col('weight').sum().alias('weight_total'),
        # composition
        safe_sum_when(pl.col('product_type') == 'Food').alias('units_food'),
        safe_sum_when(pl.col('product_type') == 'Non Food').alias('units_non_food'),
        safe_sum_when(pl.col('product_type') == 'Pet Food').alias('units_pet_food'),
        safe_sum_when(pl.col('macro_category') == 'RETAIL').alias('units_RETAIL'),
        safe_sum_when(pl.col('macro_category') == 'FBA').alias('units_FBA'),
        safe_sum_when(pl.col('is_hazmat') == 'Y').alias('units_hazmat'),
        # outcome: all non-Sales = entered funnel
        safe_sum_when(pl.col('recovery_type') != 'Sales').alias('units_recovered'),
        safe_sum_when(pl.col('recovery_type') == 'Sales').alias('units_sales'),
        safe_sum_when(pl.col('recovery_type') == 'Remove Return').alias('units_remove_return'),
        safe_sum_when(pl.col('recovery_type') == 'Donations').alias('units_donations'),
        safe_sum_when(pl.col('recovery_type') == 'Bintool Donations').alias('units_bintool_donations'),
        safe_sum_when(pl.col('recovery_type') == 'Warehouse Deals and G&R').alias('units_warehouse_deals_and_gr'),
        safe_sum_when(pl.col('recovery_type') == 'Liquidations').alias('units_liquidations'),
        safe_sum_when(pl.col('recovery_type') == 'Return to Vendor').alias('units_return_to_vendor'),
        safe_sum_when(pl.col('recovery_type') == 'Bintool Theft').alias('units_bintool_theft'),
        safe_sum_when(pl.col('recovery_type') == 'Remove Liquidate').alias('units_remove_liquidate'),
        safe_sum_when(pl.col('recovery_type') == 'Bintool Remove Liquidate').alias('units_bintool_remove_liquidate'),
        # site attributes (stable within a site)
        pl.col('country').first(),
        pl.col('country_state').first(),
        pl.col('site_type').first(),
        pl.col('site_category').first(),
    ])
    .sort(['hashed_fc', 'year', 'week', 'gl_product_group'])
)
print('aggregated shape:', agg.shape)
agg.head(3)


## 3. Compute share, COGS, and site-context features

In [ ]:
# Share features and outcome rates
agg = agg.with_columns([
    # GL composition
    (pl.col('units_food')    / pl.col('units_total')).alias('share_food'),
    (pl.col('units_non_food')/ pl.col('units_total')).alias('share_non_food'),
    (pl.col('units_pet_food')/ pl.col('units_total')).alias('share_pet_food'),
    (pl.col('units_RETAIL')  / pl.col('units_total')).alias('share_RETAIL'),
    (pl.col('units_FBA')     / pl.col('units_total')).alias('share_FBA'),
    (pl.col('units_hazmat')  / pl.col('units_total')).alias('share_hazmat'),
    # primary outcome rate
    (pl.col('units_recovered') / pl.col('units_total')).alias('prob_recovered'),
    # per-outcome rates (fraction of total)
    (pl.col('units_remove_return')           / pl.col('units_total')).alias('prob_remove_return'),
    (pl.col('units_donations')               / pl.col('units_total')).alias('prob_donations'),
    (pl.col('units_bintool_donations')        / pl.col('units_total')).alias('prob_bintool_donations'),
    (pl.col('units_warehouse_deals_and_gr')   / pl.col('units_total')).alias('prob_warehouse_deals'),
    (pl.col('units_liquidations')            / pl.col('units_total')).alias('prob_liquidations'),
    (pl.col('units_return_to_vendor')         / pl.col('units_total')).alias('prob_return_to_vendor'),
    (pl.col('units_bintool_theft')            / pl.col('units_total')).alias('prob_bintool_theft'),
    (pl.col('units_remove_liquidate')         / pl.col('units_total')).alias('prob_remove_liquidate'),
    (pl.col('units_bintool_remove_liquidate') / pl.col('units_total')).alias('prob_bintool_remove_liq'),
    # COGS / intensity
    (pl.col('cogs_total')   / pl.col('units_total')).alias('avg_cogs_per_unit'),
    (pl.col('weight_total') / pl.col('units_total')).alias('avg_weight_per_unit'),
    (pl.col('cogs_total')   / pl.col('weight_total')).alias('cogs_per_unit_weight'),
])

# Conditional outcome mix (share of *recovered* units going to each outcome)
agg = agg.with_columns([
    pl.when(pl.col('units_recovered') > 0)
      .then(pl.col(col) / pl.col('units_recovered'))
      .otherwise(0.0)
      .alias(col.replace('units_', 'recov_share_'))
    for col in OUTCOME_COLS
])

# Site-week context via window expressions
agg = agg.with_columns([
    pl.col('units_total').sum().over(['hashed_fc', 'year', 'week']).alias('site_units_total_week'),
    pl.col('weight_total').sum().over(['hashed_fc', 'year', 'week']).alias('site_weight_total_week'),
    pl.col('units_recovered').sum().over(['hashed_fc', 'year', 'week']).alias('site_units_recovered_week'),
])
agg = agg.with_columns([
    (pl.col('units_total')     / pl.col('site_units_total_week')).alias('site_units_share_week'),
    (pl.col('weight_total')    / pl.col('site_weight_total_week')).alias('site_weight_share_week'),
    (pl.col('site_units_recovered_week') / pl.col('site_units_total_week')).alias('site_prob_recovered_week'),
])

assert float(agg['units_total'].min()) > 0
assert float(agg['prob_recovered'].min()) >= 0
assert float(agg['prob_recovered'].max()) <= 1
print('feature engineering done, shape:', agg.shape)
agg.head(3)


## 4. Overall funnel-entry picture

Distribution of `prob_recovered` across all site-GL-weeks. Values close to 1.0 mean almost every unit at that site×GL×week entered the funnel.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# histogram of prob_recovered
pr = agg['prob_recovered'].to_numpy()
axes[0].hist(pr, bins=60, color=BAD_COLOR, alpha=0.85)
axes[0].set_xlabel('prob_recovered (funnel-entry rate per site-GL-week)')
axes[0].set_ylabel('count')
axes[0].set_title('Distribution of funnel-entry rate')

# bucket breakdown
buckets = ['zero (=0)', '0–10%', '10–30%', '30–60%', '>60%']
counts = [
    int((agg['prob_recovered'] == 0).sum()),
    int(((agg['prob_recovered'] > 0) & (agg['prob_recovered'] <= 0.10)).sum()),
    int(((agg['prob_recovered'] > 0.10) & (agg['prob_recovered'] <= 0.30)).sum()),
    int(((agg['prob_recovered'] > 0.30) & (agg['prob_recovered'] <= 0.60)).sum()),
    int((agg['prob_recovered'] > 0.60).sum()),
]
colors = [GOOD_COLOR, '#A8C5DA', '#F0A500', '#E07030', BAD_COLOR]
axes[1].bar(buckets, counts, color=colors)
axes[1].set_title('Rows by funnel-entry rate bucket')
axes[1].set_ylabel('n rows')
axes[1].tick_params(axis='x', rotation=30)

# total units vs recovered by year
yr = agg.group_by('year').agg([
    pl.col('units_total').sum().alias('units'),
    pl.col('units_recovered').sum().alias('recovered'),
]).sort('year').to_pandas()
x = np.arange(len(yr))
axes[2].bar(x - 0.2, yr['units'],     width=0.4, label='total units', color=NEU_COLOR)
axes[2].bar(x + 0.2, yr['recovered'], width=0.4, label='units in funnel', color=BAD_COLOR)
axes[2].set_xticks(x); axes[2].set_xticklabels(yr['year'])
axes[2].set_title('Total units vs recovered by year')
axes[2].legend()

plt.tight_layout(); plt.show()
print(f"Overall volume-weighted funnel-entry rate: "
      f"{agg['units_recovered'].sum() / agg['units_total'].sum():.3%}")


## 5. Top sites by funnel-entry rate

Volume-weighted rate (`units_recovered / units_total` aggregated over all weeks). Minimum 1 000 total units to exclude noise from tiny sites.

In [ ]:
MIN_SITE_UNITS = 1_000
TOP_N = 20

site_summary = (
    agg.group_by(['hashed_fc', 'site_type', 'site_category', 'country', 'country_state'])
    .agg([
        (pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('vw_rate'),
        pl.col('units_total').sum().alias('units'),
        pl.col('units_recovered').sum().alias('units_recovered'),
        pl.col('prob_recovered').mean().alias('unweighted_mean'),
        pl.col('share_hazmat').mean().alias('mean_hazmat'),
        pl.col('share_FBA').mean().alias('mean_fba'),
        pl.col('avg_cogs_per_unit').mean().alias('mean_cogs_per_unit'),
    ])
    .filter(pl.col('units') >= MIN_SITE_UNITS)
    .sort('vw_rate', descending=True)
)

top_sites_df = site_summary.head(TOP_N).to_pandas()
top_sites_list = top_sites_df['hashed_fc'].tolist()

fig, ax = plt.subplots(figsize=(10, 8))
colors = [BAD_COLOR if r > 0.5 else (NEU_COLOR if r > 0.2 else GOOD_COLOR)
          for r in top_sites_df['vw_rate']]
bars = ax.barh(range(TOP_N), top_sites_df['vw_rate'], color=colors)
labels = [
    f"{h[:10]}… [{st} / {c}]" for h, st, c in
    zip(top_sites_df['hashed_fc'], top_sites_df['site_type'], top_sites_df['country'])
]
ax.set_yticks(range(TOP_N)); ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Volume-weighted funnel-entry rate')
ax.set_title(f'Top {TOP_N} sites — funnel-entry rate (higher = worse)')
ax.axvline(0.5, color='#333', linewidth=0.8, linestyle='--', label='50% threshold')
ax.legend()
plt.tight_layout(); plt.show()

print('Top sites table:')
print(site_summary.head(TOP_N).select([
    pl.col('hashed_fc').str.slice(0, 12).alias('site'),
    'site_type', 'site_category', 'country', 'country_state',
    pl.col('units').cast(pl.Int64),
    pl.col('units_recovered').cast(pl.Int64),
    pl.col('vw_rate').round(4),
    pl.col('mean_hazmat').round(3),
    pl.col('mean_fba').round(3),
]))


## 6. Top GLs by funnel-entry rate

Same volume-weighted ranking, but aggregated over all sites and weeks per GL.

In [ ]:
MIN_GL_UNITS = 1_000

gl_summary = (
    agg.group_by('gl_product_group')
    .agg([
        (pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('vw_rate'),
        pl.col('units_total').sum().alias('units'),
        pl.col('units_recovered').sum().alias('units_recovered'),
        pl.col('share_hazmat').mean().alias('mean_hazmat'),
        pl.col('share_FBA').mean().alias('mean_fba'),
        pl.col('share_food').mean().alias('mean_food'),
        pl.col('avg_cogs_per_unit').mean().alias('mean_cogs_per_unit'),
    ])
    .filter(pl.col('units') >= MIN_GL_UNITS)
    .sort('vw_rate', descending=True)
)

top_gls_df = gl_summary.head(TOP_N).to_pandas()
top_gls_list = top_gls_df['gl_product_group'].tolist()

fig, ax = plt.subplots(figsize=(10, 8))
colors = [BAD_COLOR if r > 0.5 else (NEU_COLOR if r > 0.2 else GOOD_COLOR)
          for r in top_gls_df['vw_rate']]
ax.barh(range(TOP_N), top_gls_df['vw_rate'], color=colors)
ax.set_yticks(range(TOP_N))
ax.set_yticklabels(
    [f"GL {g}  [hazmat={h:.0%} FBA={f:.0%}]"
     for g, h, f in zip(top_gls_df['gl_product_group'],
                         top_gls_df['mean_hazmat'],
                         top_gls_df['mean_fba'])],
    fontsize=8
)
ax.invert_yaxis()
ax.set_xlabel('Volume-weighted funnel-entry rate')
ax.set_title(f'Top {TOP_N} GLs — funnel-entry rate (higher = worse)')
ax.axvline(0.5, color='#333', linewidth=0.8, linestyle='--')
plt.tight_layout(); plt.show()

print('Top GLs table:')
print(gl_summary.head(TOP_N).select([
    'gl_product_group',
    pl.col('units').cast(pl.Int64),
    pl.col('units_recovered').cast(pl.Int64),
    pl.col('vw_rate').round(4),
    pl.col('mean_hazmat').round(3),
    pl.col('mean_fba').round(3),
    pl.col('mean_food').round(3),
    pl.col('mean_cogs_per_unit').round(2),
]))


## 7. Outcome type stratification

Stacked bar of the **conditional recovery-type mix** (what fraction of recovered units went to each outcome) for the top-20 sites and top-20 GLs. This tells stakeholders not just *how much* inventory entered the funnel but *what kind* of loss is happening — liquidations vs donations vs remove-return, etc.

In [ ]:
recov_share_cols = [c.replace('units_', 'recov_share_') for c in OUTCOME_COLS]

def plot_outcome_mix(data, id_col, title, top_list):
    sub = (
        data.filter(pl.col(id_col).is_in(top_list))
        .group_by(id_col)
        .agg(
            [pl.col('units_recovered').sum()] +
            [pl.col(c).mean().alias(c) for c in recov_share_cols]
        )
        .sort('units_recovered', descending=True)
        .to_pandas()
    )
    labels = [str(v)[:14] for v in sub[id_col]]
    fig, ax = plt.subplots(figsize=(14, 6))
    bottoms = np.zeros(len(sub))
    for col, label, color in zip(recov_share_cols, OUTCOME_LABELS, OUTCOME_COLORS):
        vals = sub[col].fillna(0).values
        ax.bar(labels, vals, bottom=bottoms, label=label, color=color)
        bottoms += vals
    ax.set_ylabel('Share of recovered units')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(loc='upper right', fontsize=7, ncol=2)
    plt.tight_layout(); plt.show()

plot_outcome_mix(agg, 'hashed_fc',       'Top-20 sites — recovery outcome mix',  top_sites_list)
plot_outcome_mix(agg, 'gl_product_group','Top-20 GLs — recovery outcome mix',    top_gls_list)


## 8. Site characteristic analysis

How do `site_type`, `site_category`, and `country` relate to funnel-entry rates? Box plots sorted by median (descending). Groups with high medians are structurally more prone to funnel entry — useful for policy-level intervention.

In [ ]:
site_vw = (
    agg.group_by(['hashed_fc', 'site_type', 'site_category', 'country'])
    .agg([
        (pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('vw_rate'),
        pl.col('units_total').sum().alias('units'),
    ])
    .filter(pl.col('units') >= MIN_SITE_UNITS)
    .to_pandas()
)

for cat_col, title in [
    ('site_type',     'Funnel-entry rate by site type'),
    ('site_category', 'Funnel-entry rate by site category'),
    ('country',       'Funnel-entry rate by country'),
]:
    order = (
        site_vw.groupby(cat_col)['vw_rate']
        .median().sort_values(ascending=False).index.tolist()
    )
    n_per = site_vw.groupby(cat_col)['hashed_fc'].count().to_dict()
    fig, ax = plt.subplots(figsize=(max(8, len(order) * 0.9), 5))
    sns.boxplot(data=site_vw, x=cat_col, y='vw_rate', order=order,
               palette='RdYlGn_r', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Volume-weighted funnel-entry rate')
    ax.tick_params(axis='x', rotation=40)
    for i, grp in enumerate(order):
        ax.text(i, -0.05, f'n={n_per.get(grp, 0)}',
                ha='center', va='top', fontsize=7, transform=ax.get_xaxis_transform())
    plt.tight_layout(); plt.show()


## 9. GL composition and COGS analysis

What share features and COGS characteristics correlate with high funnel-entry? These plots help identify *what kind* of inventory is most at risk.

In [ ]:
gl_level = (
    agg.group_by('gl_product_group')
    .agg([
        (pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('vw_rate'),
        pl.col('units_total').sum().alias('units'),
        pl.col('share_hazmat').mean().alias('mean_hazmat'),
        pl.col('share_FBA').mean().alias('mean_fba'),
        pl.col('share_food').mean().alias('mean_food'),
        pl.col('share_RETAIL').mean().alias('mean_RETAIL'),
        pl.col('avg_cogs_per_unit').mean().alias('mean_cogs'),
        pl.col('avg_weight_per_unit').mean().alias('mean_weight'),
    ])
    .filter(pl.col('units') >= MIN_GL_UNITS)
    .to_pandas()
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Hazmat share vs funnel-entry rate
sc = axes[0].scatter(gl_level['mean_hazmat'], gl_level['vw_rate'],
                     s=np.sqrt(gl_level['units']) * 0.01 + 20,
                     c=gl_level['vw_rate'], cmap='RdYlGn_r', alpha=0.7)
axes[0].set_xlabel('Mean hazmat share'); axes[0].set_ylabel('Funnel-entry rate')
axes[0].set_title('Hazmat share vs funnel-entry rate\n(bubble size ∝ volume)')
plt.colorbar(sc, ax=axes[0])

# COGS per unit vs funnel-entry rate
sc2 = axes[1].scatter(gl_level['mean_cogs'], gl_level['vw_rate'],
                      s=np.sqrt(gl_level['units']) * 0.01 + 20,
                      c=gl_level['vw_rate'], cmap='RdYlGn_r', alpha=0.7)
axes[1].set_xlabel('Mean avg_cogs_per_unit'); axes[1].set_ylabel('Funnel-entry rate')
axes[1].set_title('COGS per unit vs funnel-entry rate')
plt.colorbar(sc2, ax=axes[1])

# FBA majority vs not
gl_level['fba_majority'] = (gl_level['mean_fba'] > 0.5).map({False: 'FBA share ≤50%', True: 'FBA share >50%'})
sns.boxplot(data=gl_level, x='fba_majority', y='vw_rate',
            order=['FBA share ≤50%', 'FBA share >50%'],
            palette={'FBA share ≤50%': GOOD_COLOR, 'FBA share >50%': BAD_COLOR}, ax=axes[2])
axes[2].set_xticklabels(['FBA share ≤50%', 'FBA share >50%'])
axes[2].set_ylabel('Funnel-entry rate'); axes[2].set_xlabel('')
axes[2].set_title('FBA-majority GLs vs others')
plt.tight_layout(); plt.show()


In [ ]:
# Correlation heatmap between share/COGS features and prob_recovered
num_cols = [
    'prob_recovered', 'share_hazmat', 'share_FBA', 'share_food',
    'share_non_food', 'share_pet_food', 'share_RETAIL',
    'avg_cogs_per_unit', 'avg_weight_per_unit', 'cogs_per_unit_weight',
    'site_units_share_week',
]
corr = agg.select(num_cols).to_pandas().corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.4, ax=ax)
ax.set_title('Correlation: share/COGS features vs prob_recovered (site-GL-week level)')
plt.tight_layout(); plt.show()


## 10. Seasonal analysis

**Key question:** Is funnel-entry rate uniform across the year, or does it spike in certain weeks or months? Seasonal spikes can be driven by promotions, inventory clearance cycles, or operational capacity constraints.

Three views:
1. **Overall weekly trend** across all years — does the pattern repeat?
2. **Monthly heatmap** by `site_type` and by `country` — which groups are most seasonal and when?
3. **Top-5 worst sites** monthly profile — do high-loss sites spike together or at different times (suggesting systemic vs local causes)?

In [ ]:
# Weekly volume-weighted mean prob_recovered, averaged across all site-GL pairs.
weekly = (
    agg.group_by(['year', 'week'])
    .agg([
        (pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('vw_rate'),
        pl.col('units_total').sum().alias('units'),
    ])
    .sort(['year', 'week'])
    .to_pandas()
)
weekly['week_idx'] = range(len(weekly))

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(weekly['week_idx'], weekly['vw_rate'], color=BAD_COLOR, linewidth=1.2)
ax.fill_between(weekly['week_idx'], 0, weekly['vw_rate'], color=BAD_COLOR, alpha=0.2)
# Annotate year boundaries
for yr in weekly['year'].unique():
    idx = weekly.loc[weekly['year'] == yr, 'week_idx'].iloc[0]
    ax.axvline(idx, color='#888', linewidth=0.7, linestyle='--')
    ax.text(idx + 0.5, ax.get_ylim()[1] * 0.97, str(yr), fontsize=8, color='#555')
ax.set_xlabel('Week index'); ax.set_ylabel('Volume-weighted funnel-entry rate')
ax.set_title('Overall weekly funnel-entry rate across all years')
plt.tight_layout(); plt.show()


In [ ]:
# Monthly mean funnel-entry rate by site_type — heatmap (site_type × month)
monthly_type = (
    agg.group_by(['site_type', 'month'])
    .agg(
        (pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('vw_rate')
    )
    .sort(['site_type', 'month'])
    .to_pandas()
    .pivot(index='site_type', columns='month', values='vw_rate')
    .fillna(0)
)
monthly_type.columns = [f'M{c:02d}' for c in monthly_type.columns]

fig, ax = plt.subplots(figsize=(13, max(4, len(monthly_type) * 0.55)))
sns.heatmap(monthly_type, annot=True, fmt='.2f', cmap='RdYlGn_r',
            linewidths=0.3, ax=ax, vmin=0, vmax=1)
ax.set_title('Monthly funnel-entry rate by site type\n(red = high loss)')
ax.set_xlabel('Month'); ax.set_ylabel('Site type')
plt.tight_layout(); plt.show()


In [ ]:
# Monthly mean by country — heatmap
monthly_country = (
    agg.group_by(['country', 'month'])
    .agg(
        (pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('vw_rate')
    )
    .sort(['country', 'month'])
    .to_pandas()
    .pivot(index='country', columns='month', values='vw_rate')
    .fillna(0)
)
monthly_country.columns = [f'M{c:02d}' for c in monthly_country.columns]

fig, ax = plt.subplots(figsize=(13, max(4, len(monthly_country) * 0.55)))
sns.heatmap(monthly_country, annot=True, fmt='.2f', cmap='RdYlGn_r',
            linewidths=0.3, ax=ax, vmin=0, vmax=1)
ax.set_title('Monthly funnel-entry rate by country\n(red = high loss)')
ax.set_xlabel('Month'); ax.set_ylabel('Country')
plt.tight_layout(); plt.show()


In [ ]:
# Week-of-year profile (averaged across all years) — shows repeating seasonality
week_profile = (
    agg.group_by('week')
    .agg(
        (pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('vw_rate')
    )
    .sort('week')
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(week_profile['week'], week_profile['vw_rate'],
       color=[BAD_COLOR if r > week_profile['vw_rate'].median() else GOOD_COLOR
              for r in week_profile['vw_rate']])
ax.axhline(week_profile['vw_rate'].median(), color='#333', linewidth=1,
           linestyle='--', label='median')
ax.set_xlabel('ISO week of year'); ax.set_ylabel('Volume-weighted funnel-entry rate')
ax.set_title('Seasonality: funnel-entry rate by ISO week (averaged across all years)\n'
             'red = above median | green = below median')
ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# Top-5 worst sites: monthly actual vs overall monthly average
top5_sites = site_summary.head(5)['hashed_fc'].to_list()

overall_monthly = (
    agg.group_by('month')
    .agg((pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('overall'))
    .sort('month')
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(overall_monthly['month'], overall_monthly['overall'],
        color='#333', linewidth=2, linestyle='--', label='all sites avg')
palette = plt.cm.Set1(np.linspace(0, 0.8, 5))
for site, color in zip(top5_sites, palette):
    s = (
        agg.filter(pl.col('hashed_fc') == site)
        .group_by('month')
        .agg((pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('vw'))
        .sort('month').to_pandas()
    )
    ax.plot(s['month'], s['vw'], marker='o', color=color,
            label=f"{site[:10]}…", linewidth=1.4)
ax.set_xlabel('Month'); ax.set_ylabel('Volume-weighted funnel-entry rate')
ax.set_title('Top-5 worst sites — monthly profile vs overall average')
ax.legend(fontsize=8); ax.set_xticks(range(1, 13))
plt.tight_layout(); plt.show()


In [ ]:
# Top-5 worst GLs: monthly profile
top5_gls = gl_summary.head(5)['gl_product_group'].to_list()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(overall_monthly['month'], overall_monthly['overall'],
        color='#333', linewidth=2, linestyle='--', label='all GLs avg')
for gl, color in zip(top5_gls, palette):
    g = (
        agg.filter(pl.col('gl_product_group') == gl)
        .group_by('month')
        .agg((pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('vw'))
        .sort('month').to_pandas()
    )
    ax.plot(g['month'], g['vw'], marker='s', color=color,
            label=f"GL {gl}", linewidth=1.4)
ax.set_xlabel('Month'); ax.set_ylabel('Volume-weighted funnel-entry rate')
ax.set_title('Top-5 worst GLs — monthly profile vs overall average')
ax.legend(fontsize=8); ax.set_xticks(range(1, 13))
plt.tight_layout(); plt.show()


## 11. Intervention priority matrix (site × GL)

Two dimensions matter for prioritising action:
- **Rate** (`vw_rate`): how bad the problem is per unit processed — indicates a structural issue.
- **Volume** (`units_recovered`): absolute number of units lost — indicates the scale of the business cost.

The top-right quadrant (high rate + high volume) is Priority 1. Top-left (high rate, low volume) = investigate root cause but lower urgency. Bottom-right (low rate, high volume) = monitor for trend deterioration.

In [ ]:
MIN_SG_UNITS = 500

site_gl_summary = (
    agg.group_by(['hashed_fc', 'gl_product_group', 'site_type', 'country'])
    .agg([
        (pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('vw_rate'),
        pl.col('units_total').sum().alias('units'),
        pl.col('units_recovered').sum().alias('units_recovered'),
        pl.col('share_hazmat').mean().alias('mean_hazmat'),
        pl.col('avg_cogs_per_unit').mean().alias('mean_cogs_per_unit'),
        pl.col('share_FBA').mean().alias('mean_fba'),
    ])
    .filter(pl.col('units') >= MIN_SG_UNITS)
    .sort('units_recovered', descending=True)
)

print('Top 30 site-GL pairs by total units lost to recovery funnel:')
print(site_gl_summary.head(30).select([
    pl.col('hashed_fc').str.slice(0, 12).alias('site'),
    'gl_product_group', 'site_type', 'country',
    pl.col('units').cast(pl.Int64),
    pl.col('units_recovered').cast(pl.Int64),
    pl.col('vw_rate').round(4),
    pl.col('mean_hazmat').round(3),
    pl.col('mean_cogs_per_unit').round(2),
]))


In [ ]:
sg_pd = site_gl_summary.to_pandas()
rate_median   = sg_pd['vw_rate'].median()
volume_median = sg_pd['units_recovered'].median()

fig, ax = plt.subplots(figsize=(12, 8))
sc = ax.scatter(
    sg_pd['vw_rate'],
    sg_pd['units_recovered'],
    s=np.sqrt(sg_pd['units']) * 0.05 + 5,
    c=sg_pd['vw_rate'],
    cmap='RdYlGn_r',
    alpha=0.6,
    linewidths=0,
)
plt.colorbar(sc, ax=ax, label='vw funnel-entry rate')
ax.axvline(rate_median,   color='#555', linewidth=1, linestyle='--')
ax.axhline(volume_median, color='#555', linewidth=1, linestyle='--')
ax.set_xlabel('Volume-weighted funnel-entry rate (higher = worse)')
ax.set_ylabel('Total units recovered (scale of loss)')
ax.set_title('Site × GL intervention priority matrix\n'
             'top-right = high rate AND high volume → Priority 1')

for txt, x, y in [
    ('← lower rate\nhigh volume\n(monitor)', 0.02, volume_median * 3),
    ('→ high rate\nhigh volume\n(Priority 1)', rate_median * 1.05, volume_median * 3),
    ('high rate\nlow volume\n(investigate)', rate_median * 1.05, volume_median * 0.1),
]:
    ax.text(x, y, txt, fontsize=8, color='#444', style='italic')

plt.tight_layout(); plt.show()


## 12. Seasonal pattern by characteristic

Does the seasonality of funnel-entry differ across site types and GLs? A site type that spikes in Q4 may need pre-holiday capacity intervention, while one that spikes mid-year might reflect inventory clearance cycles.

In [ ]:
# Quarter-level funnel-entry rate by site_type
agg_q = agg.with_columns(
    pl.when(pl.col('month') <= 3).then(pl.lit('Q1'))
     .when(pl.col('month') <= 6).then(pl.lit('Q2'))
     .when(pl.col('month') <= 9).then(pl.lit('Q3'))
     .otherwise(pl.lit('Q4'))
     .alias('quarter')
)

qt_type = (
    agg_q.group_by(['site_type', 'quarter'])
    .agg((pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('vw_rate'))
    .sort(['site_type', 'quarter'])
    .to_pandas()
    .pivot(index='site_type', columns='quarter', values='vw_rate')
    [['Q1', 'Q2', 'Q3', 'Q4']].fillna(0)
)

fig, ax = plt.subplots(figsize=(10, max(4, len(qt_type) * 0.6)))
sns.heatmap(qt_type, annot=True, fmt='.2f', cmap='RdYlGn_r',
            linewidths=0.3, ax=ax, vmin=0, vmax=1)
ax.set_title('Quarterly funnel-entry rate by site type\n'
             'which site types worsen in which quarter?')
plt.tight_layout(); plt.show()


In [ ]:
# Quarter-level for top GLs
qt_gl = (
    agg_q.filter(pl.col('gl_product_group').is_in(top_gls_list))
    .group_by(['gl_product_group', 'quarter'])
    .agg((pl.col('units_recovered').sum() / pl.col('units_total').sum()).alias('vw_rate'))
    .sort(['gl_product_group', 'quarter'])
    .to_pandas()
    .pivot(index='gl_product_group', columns='quarter', values='vw_rate')
    [['Q1', 'Q2', 'Q3', 'Q4']].fillna(0)
)
qt_gl.index = [f'GL {i}' for i in qt_gl.index]

fig, ax = plt.subplots(figsize=(10, max(4, len(qt_gl) * 0.6)))
sns.heatmap(qt_gl, annot=True, fmt='.2f', cmap='RdYlGn_r',
            linewidths=0.3, ax=ax, vmin=0, vmax=1)
ax.set_title('Quarterly funnel-entry rate — top-20 worst GLs\n'
             'which GLs are seasonal vs structurally high?')
plt.tight_layout(); plt.show()


## 13. Intervention shortlist

Top 15 site-GL pairs ranked by **absolute units lost to the recovery funnel** (the actual business cost). Use this table as the starting point for site-level investigations.

_Fill in observations after running:_
- Which site types / countries dominate the top rows?
- Are the worst pairs hazmat-heavy or FBA-heavy?
- Is the loss concentrated in specific quarters (seasonal) or year-round (structural)?
- Which outcome type (liquidations, donations, remove-return) dominates — and does that suggest a specific process fix?

In [ ]:
shortlist = site_gl_summary.head(15).select([
    pl.col('hashed_fc').str.slice(0, 14).alias('site'),
    'gl_product_group', 'site_type', 'country',
    pl.col('units').cast(pl.Int64).alias('total_units'),
    pl.col('units_recovered').cast(pl.Int64).alias('units_in_funnel'),
    pl.col('vw_rate').round(4).alias('funnel_entry_rate'),
    pl.col('mean_hazmat').round(3),
    pl.col('mean_fba').round(3),
    pl.col('mean_cogs_per_unit').round(2),
])
print('Intervention shortlist (top 15 by units lost to funnel):')
print(shortlist)
